# Daily Baa yield vs. dividend yield

A daily version of [FRED graph 1629621](https://fred.stlouisfed.org/graph/?graph_id=1629621):

> Market value of all US corporate equities ÷ (dividends paid ÷ Moody's Baa corporate bond yield)

**What the ratio is.** Write it as `MV / (D / Baa)`. Rearranged, that is `Baa × MV / D`. `D / MV` is the dividend yield, so `MV / D` is 1 ÷ dividend yield, and the ratio becomes

$$\text{ratio} = 100 \times \frac{\text{Baa yield (\%)}}{\text{dividend yield (\%)}}$$

At 100 the corporate bond yields exactly what stocks pay in dividends. Above 100, bonds pay more.

**Why FRED's version is annual.** Two of its three inputs are annual: the Fed's Financial Accounts market value and the dividends series. Only the Baa yield is published more often.

**How the daily version is built.**
- Baa yield: `DBAA`, the daily series from the same Moody's release.
- Dividend yield: the SPY ETF's last four quarterly dividends divided by its daily closing price (Yahoo Finance via `yfinance`). SPY data starts in 1993.

The price and the bond yield change every trading day. The dividend total changes only when SPY pays, four times a year. That step pattern is how dividends actually arrive, not an artifact of the method.

**The S&P 500 does not match FRED's coverage.** It counts only large companies' dividends. FRED's dividend series counts dividends paid by *all sectors*, so the two dividend yields differ and the two ratios sit at different levels. The notebook therefore shows:
1. **The raw S&P 500 ratio.** 100 = Baa yield equals the S&P 500 dividend yield. Nothing is fitted.
2. **The S&P 500 ratio rescaled to FRED's level.** It uses the ratio between the two series measured in overlapping years, and the notebook prints how stable that ratio has been. If it drifts, the rescaled line is only approximate.

Needs a free [FRED API key](https://fredaccount.stlouisfed.org/apikey) saved in Colab Secrets as `FRED`, the same key as the productivity-pay-gap notebook.

**Licensing.** Moody's licence prohibits redistributing its yield data. Keep any outputs built on `DBAA`/`BAA` for personal use unless you have permission.

In [ ]:
%pip install -q fredapi yfinance

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from fredapi import Fred
import yfinance as yf

#FRED API key: Colab Secrets (name: FRED), or the FRED_API_KEY environment variable when running locally
try:
    from google.colab import userdata
    FRED_KEY = userdata.get('FRED')
except ImportError:
    FRED_KEY = os.environ['FRED_API_KEY']
fred = Fred(api_key=FRED_KEY)

### 1. The original annual series, reproduced

FRED's graph turns the monthly `BAA` into annual averages by default. This cell does the same, then prints the 1958 and latest values so you can check them against the screenshot (about 100 near 1958 and about 200 in the latest year).

In [ ]:
MV_ID  = 'BOGZ1LM893064105A'  #All Domestic Sectors; Corporate Equities; Liability, Market Value Levels (annual, $ millions)
DIV_ID = 'ASDIPAA027N'        #All Sectors; Dividends Paid (IMA), Transactions (annual, $ millions SAAR)
MV_TITLE = 'All Domestic Sectors; Corporate Equities; Liability, Market Value Levels'

def get_annual_mv():
    #falls back to a title search in case FRED renames the series ID
    try:
        return fred.get_series(MV_ID), MV_ID
    except Exception:
        hits = fred.search(MV_TITLE)
        hits = hits[(hits['title'] == MV_TITLE) & (hits['frequency_short'] == 'A')]
        if hits.empty:
            raise ValueError('Market value series not found; look up its ID on FRED and set MV_ID')
        sid = hits.index[0]
        return fred.get_series(sid), sid

mv, mv_id = get_annual_mv()
div = fred.get_series(DIV_ID)
baa_m = fred.get_series('BAA')
print('Market value series:', mv_id, '-', fred.get_series_info(mv_id)['title'])

annual = pd.DataFrame({'mv': mv, 'div': div})
annual.index = annual.index.year
annual['baa'] = baa_m.groupby(baa_m.index.year).mean()
annual = annual.dropna()
annual['fred_ratio'] = annual['mv'] / (annual['div'] / annual['baa'])
annual['div_yield_pct'] = 100 * annual['div'] / annual['mv']

print(annual.loc[[y for y in (1950, 1958, 2000) if y in annual.index] + [annual.index[-1]]].round(2))

### 2. Daily inputs

In [ ]:
dbaa = fred.get_series('DBAA').dropna().rename('baa')

spy = yf.Ticker('SPY').history(period='max', auto_adjust=False, actions=True)
spy.index = pd.to_datetime(spy.index).tz_localize(None).normalize()

#trailing-12-month dividends = sum of the last 4 payments (a fixed 365-day window
#sometimes catches 5 payments, because ex-dates fall on the third Friday of the month)
pays = spy.loc[spy['Dividends'] > 0, 'Dividends']
gaps = pays.index.to_series().diff().dt.days.dropna()
print(f'SPY payments: {len(pays)}, median days between them: {gaps.median():.0f} (expect ~91)')
ttm = pays.rolling(4).sum().reindex(spy.index).ffill()

daily = pd.DataFrame({'price': spy['Close'], 'ttm_div': ttm})
daily = daily.join(dbaa, how='left')
daily['baa'] = daily['baa'].ffill(limit=5)    #FRED and NYSE holidays differ
daily = daily.dropna()
daily['div_yield_pct'] = 100 * daily['ttm_div'] / daily['price']
daily['raw_ratio'] = 100 * daily['baa'] / daily['div_yield_pct']
print(daily.tail().round(3))

### 3. Measuring the gap to FRED's level

For each year this builds an annual figure from the daily data that matches FRED's timing: the price on the last trading day (FRED's market value is end of year), trailing dividends at that date, and the year's average Baa yield. It then compares that figure with FRED's annual ratio. The scale factor used below is the median of the latest 5 overlapping years. The spread across all years shows how far to trust it.

In [ ]:
last_day = daily.groupby(daily.index.year).tail(1)
comp = pd.DataFrame({
    'price_end': last_day['price'].values,
    'ttm_div_end': last_day['ttm_div'].values,
}, index=last_day.index.year)
comp['baa_avg'] = daily['baa'].groupby(daily.index.year).mean()
comp['spy_ratio'] = comp['baa_avg'] * comp['price_end'] / comp['ttm_div_end']
comp = comp.join(annual[['fred_ratio']], how='inner')
comp = comp[comp.index > daily.index[0].year]   #first year has a partial dividend history
comp['scale'] = comp['fred_ratio'] / comp['spy_ratio']

SCALE = comp['scale'].tail(5).median()
print(comp[['spy_ratio', 'fred_ratio', 'scale']].round(3).to_string())
print(f'\nscale factor (median of last 5 years): {SCALE:.3f}')
print(f'scale across all {len(comp)} years: min {comp.scale.min():.3f}, max {comp.scale.max():.3f}, '
      f'coefficient of variation {comp.scale.std() / comp.scale.mean():.1%}')

daily['scaled_ratio'] = daily['raw_ratio'] * SCALE

### 4. Charts

In [ ]:
BLUE, ORANGE, INK, MUTED = '#2a78d6', '#eb6834', '#0b0b0b', '#52514e'
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': MUTED, 'axes.labelcolor': INK,
                     'xtick.color': MUTED, 'ytick.color': MUTED,
                     'axes.grid': True, 'grid.color': '#e4e3df', 'grid.linewidth': 0.6})

latest = daily.index[-1]
fig, axes = plt.subplots(2, 1, figsize=(11, 9), sharex=False)

ax = axes[0]
ax.plot(daily.index, daily['raw_ratio'], color=BLUE, lw=1.2, label='S&P 500 (SPY), daily')
ax.axhline(100, color=INK, ls='--', lw=1)
ax.set_title(f'Raw: 100 × Baa yield ÷ S&P 500 dividend yield   (latest {latest:%Y-%m-%d}: {daily.raw_ratio.iloc[-1]:.0f})',
             loc='left', fontsize=11, color=INK)
ax.text(daily.index[0], 103, 'Baa yield = S&P 500 dividend yield', color=MUTED, fontsize=9, va='bottom')

ax = axes[1]
ax.plot(daily.index, daily['scaled_ratio'], color=BLUE, lw=1.2, label=f'Daily S&P 500 version × {SCALE:.2f}')
fred_pts = annual.loc[annual.index >= daily.index[0].year, 'fred_ratio']
ax.plot(pd.to_datetime(fred_pts.index.astype(str) + '-12-31'), fred_pts, 'o', ms=6, color=ORANGE,
        markeredgecolor='white', markeredgewidth=1.5, label='FRED annual (original graph)')
ax.axhline(100, color=INK, ls='--', lw=1)
ax.set_title(f'Rescaled to FRED level   (latest {latest:%Y-%m-%d}: {daily.scaled_ratio.iloc[-1]:.0f})',
             loc='left', fontsize=11, color=INK)
ax.legend(frameon=False, loc='best')

for a in axes:
    a.xaxis.set_major_locator(mdates.YearLocator(5))
    a.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig.tight_layout()
plt.show()

In [ ]:
#last two years, where the day-to-day movement is visible
recent = daily.loc[latest - pd.DateOffset(years=2):]
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(recent.index, recent['scaled_ratio'], color=BLUE, lw=1.5)
ax.set_title('Rescaled daily ratio, last 2 years', loc='left', fontsize=11, color=INK)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
fig.tight_layout()
plt.show()

print(f'Latest ({latest:%Y-%m-%d}): Baa {daily.baa.iloc[-1]:.2f}%, '
      f'S&P 500 dividend yield {daily.div_yield_pct.iloc[-1]:.2f}%, '
      f'raw ratio {daily.raw_ratio.iloc[-1]:.0f}, rescaled {daily.scaled_ratio.iloc[-1]:.0f}')

### 5. Adding buybacks

Companies return cash two ways: dividends, and buying back their own shares. Since the late 1990s US companies have returned more through buybacks than through dividends, so a ratio built on dividends alone overstates how little cash stocks pay compared with bonds.

**Data.** No free source publishes buybacks daily. S&P's quarterly buyback figures are a licensed spreadsheet with no API. This uses the Fed's Financial Accounts on FRED: *net* equity issuance by nonfinancial corporations, quarterly, at an annual rate. Net issuance = new shares sold − shares bought back − shares retired for cash in takeovers. Its negative is the net cash returned to shareholders. Shares issued to employees are subtracted, which is why the net figure is well below headline gross buyback totals.

**What it leaves out.**
- **Financial companies' buybacks.** Banks and insurers are a large share of them.
- **Negative years.** In years when companies issued more shares than they retired, such as the 1950s–70s, the net buyback yield is negative and *lowers* the total shareholder yield.

**Timing.** Each quarter is used only from 75 days after it ends, roughly when the Fed publishes it. That way the daily line never uses numbers nobody had yet. 75 days is an approximation of the publication date, not the exact release date.

**Daily construction.**
- Net buyback yield = last 4 quarters of net buybacks ÷ nonfinancial equity market value.
- The market value is the latest published quarterly value, moved each day by SPY's price change since that quarter ended.
- The S&P 500 dividend yield is put on the FRED dividend basis by dividing it by the scale factor from section 3. This makes the dividend part match the original graph before buybacks are added.

$$\text{ratio} = 100 \times \frac{\text{Baa yield}}{\text{dividend yield} + \text{net buyback yield}}$$

In [ ]:
def find_series(title, freq, prefer_sa):
    #FRED's Financial Accounts IDs are opaque, so look the series up by exact title
    hits = fred.search(title)
    hits = hits[(hits['title'].str.strip().str.lower() == title.lower()) & (hits['frequency_short'] == freq)]
    if hits.empty:
        raise ValueError(f'No {freq} series titled {title!r}; look it up on FRED and pass the ID to fred.get_series')
    pref = hits[hits['seasonal_adjustment_short'] == prefer_sa]
    sid = (pref if not pref.empty else hits).index[0]
    print(f'{sid}: {title} ({freq}, {hits.loc[sid, "seasonal_adjustment_short"]})')
    return fred.get_series(sid).dropna()

net_iss = find_series('Nonfinancial Corporate Business; Corporate Equities; Liability, Transactions', 'Q', 'SAAR')
nf_mv   = find_series('Nonfinancial Corporate Business; Corporate Equities; Liability, Level', 'Q', 'NSA')

#annual version of the original FRED graph, with buybacks added to dividends
bb_q = -net_iss                                     #net buybacks, $ millions at an annual rate
yr = bb_q.groupby(bb_q.index.year)
annual['buybacks'] = yr.mean()[yr.count() == 4]     #mean of 4 annualized quarters = the year's flow; full years only
annual['bb_yield_pct'] = 100 * annual['buybacks'] / annual['mv']
annual['sh_yield_pct'] = annual['div_yield_pct'] + annual['bb_yield_pct']
annual['fred_bb_ratio'] = (100 * annual['baa'] / annual['sh_yield_pct']).where(annual['sh_yield_pct'] > 0)
print(annual[['div_yield_pct', 'bb_yield_pct', 'fred_ratio', 'fred_bb_ratio']].iloc[::10].round(2).to_string())

#daily net buyback yield, each quarter used only once it would have been published
LAG_DAYS = 75
q = pd.DataFrame({'bb_ttm': bb_q.rolling(4).mean(), 'mv_q': nf_mv}).dropna()
q['q_end'] = q.index.to_period('Q').end_time.normalize()
q['price_q_end'] = daily['price'].reindex(q['q_end'], method='ffill').values
q['available'] = q['q_end'] + pd.Timedelta(days=LAG_DAYS)
q = q.dropna().sort_values('available')

d = pd.merge_asof(daily[['price']].rename_axis('date').reset_index(), q[['available', 'bb_ttm', 'mv_q', 'price_q_end']],
                  left_on='date', right_on='available').set_index('date')
daily['bb_yield_pct'] = 100 * d['bb_ttm'] / (d['mv_q'] * d['price'] / d['price_q_end'])

#dividend yield on the FRED all-sector basis (ratio_fred / ratio_spy = dy_spy / dy_fred, so dy_fred = dy_spy / SCALE)
daily['div_yield_fred_basis'] = daily['div_yield_pct'] / SCALE
daily['sh_yield_pct'] = daily['div_yield_fred_basis'] + daily['bb_yield_pct']
daily['bb_ratio'] = (100 * daily['baa'] / daily['sh_yield_pct']).where(daily['sh_yield_pct'] > 0)
print(daily[['baa', 'div_yield_fred_basis', 'bb_yield_pct', 'scaled_ratio', 'bb_ratio']].dropna().tail().round(2))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 9))

ax = axes[0]
a = annual.dropna(subset=['fred_ratio'])
ax.plot(a.index, a['fred_ratio'], color=BLUE, lw=2, label='Dividends only (original FRED graph)')
ax.plot(a.index, a['fred_bb_ratio'], color=ORANGE, lw=2, label='Dividends + net buybacks')
ax.axhline(100, color=INK, ls='--', lw=1)
ax.set_title('Annual: 100 × Baa yield ÷ cash yield to shareholders', loc='left', fontsize=11, color=INK)
ax.legend(frameon=False, loc='best')

ax = axes[1]
dd = daily.dropna(subset=['bb_ratio'])
ax.plot(dd.index, dd['scaled_ratio'], color=BLUE, lw=1.2, label='Dividends only')
ax.plot(dd.index, dd['bb_ratio'], color=ORANGE, lw=1.2, label='Dividends + net buybacks')
ax.axhline(100, color=INK, ls='--', lw=1)
ax.set_title(f'Daily, FRED basis   (latest {dd.index[-1]:%Y-%m-%d}: '
             f'{dd.scaled_ratio.iloc[-1]:.0f} dividends only, {dd.bb_ratio.iloc[-1]:.0f} with buybacks)',
             loc='left', fontsize=11, color=INK)
ax.legend(frameon=False, loc='best')
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig.tight_layout()
plt.show()

last = dd.iloc[-1]
print(f'Latest ({dd.index[-1]:%Y-%m-%d}): Baa {last.baa:.2f}%, dividend yield (FRED basis) {last.div_yield_fred_basis:.2f}%, '
      f'net buyback yield {last.bb_yield_pct:.2f}% (data through {q.loc[q.available <= dd.index[-1], "q_end"].iloc[-1]:%Y-%m-%d})')